In [100]:
import os
import json
import random
from pathlib import Path
import torch
from torch.utils.data import Dataset

class MelDataset(Dataset):
    def __init__(self, data_dir, segment_len=128):
        self.data_dir = data_dir
        self.segment_len = segment_len
        
        # Load the mapping from speakers' name to their ids
        mapping_path = Path(self.data_dir) / "mapping.json"
        mapping = json.load(mapping_path.open())
        self.speaker2id = mapping["speaker2id"]
        
        # Load the metadata of training data
        metadata_path = Path(self.data_dir) / "metadata.json"
        metadata = json.load(metadata_path.open())["speakers"]
        
        # Get the toatal number of speakers
        self.speaker_num = len(metadata.keys())
        self.data = []
        for speaker in metadata.keys():
            for utterances in metadata[speaker]:
                self.data.append([utterances["feature_path"], self.speaker2id[speaker]])
                
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        feature_path, speaker = self.data[index]
        # Load preprocessed mel-spectrogram feature
        mel = torch.load(os.path.join(self.data_dir, feature_path))
        
        # Do data segmentation, randomly segment mel-spectrogram into frames by given "segment_len"
        if len(mel) > self.segment_len:
            start = random.randint(0, len(mel) - self.segment_len)
            # Get a continuous frames by given "segment_len"
            mel = torch.FloatTensor(mel[start:start+self.segment_len])
        else:
            mel = torch.FloatTensor(mel)
        
        speaker = torch.FloatTensor([speaker]).long()
        
        return mel, speaker
    
    def get_speaker_number(self):
        return self.speaker_num

In [101]:
from torch.utils.data import DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence

def collate_batch(batch):
    """
    Process features in a batch
    """
    mel, speaker = zip(*batch)
    # Pad mel-spectrograms into same length, which is the segment_len
    mel = pad_sequence(mel, batch_first=True, padding_value=-20)
    # shape: (batch_size, segment_len, 40)
    return mel, torch.FloatTensor(speaker).long()

def get_dataloader(data_dir, batch_size, n_workers):
    dataset = MelDataset(data_dir)
    speaker_num = dataset.get_speaker_number()
    
    # Split the dataset into training set and validation set
    training_len = int(0.9 * len(dataset))
    lengths = [training_len, len(dataset) - training_len]
    training_set, validation_set = random_split(dataset, lengths)
    
    training_loader = DataLoader(
        training_set,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=n_workers,
        pin_memory=True,
        collate_fn=collate_batch,
    )
    
    validation_loader = DataLoader(
        validation_set,
        batch_size=batch_size,
        drop_last=True,
        num_workers=n_workers,
        pin_memory=True,
        collate_fn=collate_batch,
    )
    
    return training_loader, validation_loader, speaker_num

In [102]:
# Define model with encoder-only architecture
import torch.nn as nn

class Classifier(nn.Module):
    def __init__(self, d_model=80, n_spks=600, dropout=0.1):
        super().__init__()
        # Transform dimension of mel-spectrograms into input dimension of the model (d_model)
        self.prenet = nn.Linear(40, d_model)
        
        self.encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=1, dim_feedforward=256, dropout=dropout
        )
        
        self.encoder = nn.TransformerEncoder(
            encoder_layer=self.encoder_layer, num_layers=8
        )
        
        # Transform dimension of model output into number of speakers
        self.pred_layer = nn.Sequential(
            nn.Linear(d_model, n_spks)
        )
        
    def forward(self, mels):
        """
        input shape: (batch_size, segment_length, 40)
        
        output shape: (batch_size, n_spks)
        """
        out = self.prenet(mels) # shape: (batch_size, segment_length, d_model)
        
        out = out.permute(1, 0, 2) # shape: (segment_length, batch_size, d_model)
        # The encoder layers except the features in the shape of (length, batch_size, d_model)
        # out = self.encoder_layer(out)
        out = self.encoder(out)
        out = out.transpose(0, 1) # shape: (batch_size, segment_length, d_model)
        
        # Mean pooling
        stats = out.mean(dim=1)
        
        out = self.pred_layer(stats) # shape: (batch_size, n_spks)
        
        return out

In [103]:
# Learning rate scheduler
import math
from torch.optim import Optimizer
from torch.optim.lr_scheduler import LambdaLR

def get_cosine_scheduler_with_warmup(
    optimizer: Optimizer,
    num_warmup_steps: int,
    num_training_steps: int,
    num_cycles: float = 0.5,
    last_epoch: int = -1,
):
    """
    Create a learning rate scheduler which increases learning rate from 0 to max learning rate linearly in a period named warmup,
    then decreases learning rate from max learning rate to 0 by cosine decay
    
    Args:
        optimizer (torch.optim.Optimizer): a instance of Oprimizer
        num_warmup_steps (int): the number of steps for the warmup period
        num_training_steps (int): the total number of training steps
        num_cycles (float, optional): number of consine cycles. Defaults to 0.5, that means a half-consine cycle.
        last_epoch (int, optional): the index of the last epoch when resuming training. Defaults to -1.
        
    Return:
        scheduler: the learning rate scheduler based on torch.optim.lr_scheduler.LambdaLR
    """
    
    def lr_lambda(current_step):
        # Warmup
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        
        # cosine decay
        decay_progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
        # To transfrom value of cosine from [-1, 1] to [0, 1], we need a affine transformation: (1 + cos(x)) / 2
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * float(num_cycles) * 2.0 * decay_progress)))
    
    return LambdaLR(optimizer, lr_lambda, last_epoch)

In [104]:
def model_forward(batch, model, criterion, device):
    """
    Forward a batch through the model
    """
    mels, labels = batch
    mels, labels = mels.to(device), labels.to(device)
    
    output = model(mels)
    
    loss = criterion(output, labels)
    
    # Get speaker id with highest probability
    preds = output.argmax(1)
    # Compute mean accuract
    acc = torch.mean((preds == labels).float())
    
    return loss, acc

In [105]:
from tqdm.auto import tqdm

def validate(dataloader, model, criterion, device):
    model.eval()
    running_loss = 0.0
    running_acc = 0.0
    
    pbar = tqdm(total=len(dataloader.dataset), desc="Validation", unit=" uttr")
    
    for i, batch in enumerate(dataloader):
        with torch.no_grad():
            loss, acc = model_forward(batch, model, criterion, device)
            running_loss += loss.item()
            running_acc += acc.item()
        
        pbar.update(dataloader.batch_size)
        pbar.set_postfix(
            loss=f"{running_loss / (i+1):.3f}",
            accuracy=f"{running_acc / (i+1): .3f}"
        )
    
    pbar.close()        
    model.train()
    
    return running_acc / len(dataloader) # Mean validation accuracy

In [106]:
from torch.optim import AdamW

def get_config():
    config = {
        "data_dir": "./Dataset",
        "save_path": "./model.ckpt",
        "n_workers": 8,
        "batch_size": 32,
        "learning_rate": 1e-3,
        "validation_interval": 2000,
        "warmup_steps": 1000,
        "checkpoint_interval": 10000,
        "total_steps": 70000
    }
    
    return config

def main(
    data_dir,
    save_path,
    n_workers,
    batch_size,
    learning_rate,
    validation_interval,
    warmup_steps,
    checkpoint_interval,
    total_steps
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[Info] Device: {device}")
    
    training_loader, validation_loader, num_speakers = get_dataloader(data_dir, batch_size, n_workers)
    training_iterator = iter(training_loader)
    print("[Info] Finish loading data")
    
    model = Classifier(n_spks=num_speakers).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=learning_rate)
    scheduler = get_cosine_scheduler_with_warmup(optimizer, warmup_steps, total_steps)
    print("[Info] Create a model")
    
    best_acc = -1.0
    best_state_dict = None
    
    pbar = tqdm(total=validation_interval, desc="Trianing", unit=" step")
    
    for step in range(total_steps):
        try:
            batch = next(training_iterator)
        except StopIteration: # An epoch has finished, get a new iterator
            training_iterator = iter(training_loader)
            batch = next(training_iterator)
            
        loss, acc = model_forward(batch, model, criterion, device)
        batch_loss = loss.item()
        batch_acc = acc.item()
        
        # Update parameters
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        
        # Log
        pbar.update()
        pbar.set_postfix(
            loss=f"{batch_loss:.3f}",
            accuracy=f"{batch_acc:.3f}",
            step=step+1
        )
        
        # If number of training steps has reached the validation interval, then do validation
        if (step + 1) % validation_interval == 0:
            pbar.close()
            
            validation_acc = validate(validation_loader, model, criterion, device)
            
            # If the model has improved, save its state dict
            if validation_acc > best_acc:
                best_acc = validation_acc
                best_state_dict = model.state_dict()
                
            pbar = tqdm(total=validation_interval, desc="Trianing", unit=" step")
            
        # If number of training steps has reached the checkpoint interval, then svae current best model
        if (step + 1) % checkpoint_interval == 0 and best_state_dict is not None:
            torch.save(best_state_dict, save_path)
            pbar.write(f"Step {step + 1}, best model saved (accuracy={best_acc:.3f})")
            
    pbar.close()
    
if __name__ == "__main__":
    main(**get_config())

[Info] Device: cuda
[Info] Finish loading data
[Info] Create a model


/tmp/ipykernel_2990857/1232978545.py:14: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = nn.TransformerEncoder(


Trianing:   0%|          | 0/2000 [00:00<?, ? step/s]

Validation:   0%|          | 0/6944 [00:00<?, ? uttr/s]

Trianing:   0%|          | 0/2000 [00:00<?, ? step/s]

...

Validation:   0%|          | 0/6944 [00:00<?, ? uttr/s]

Trianing:   0%|          | 0/2000 [00:00<?, ? step/s]

Validation:   0%|          | 0/6944 [00:00<?, ? uttr/s]

Trianing:   0%|          | 0/2000 [00:00<?, ? step/s]

Step 70000, best model saved (accuracy=0.863)


In [109]:
class InferenceDataset(Dataset):
    def __init__(self, data_dir):
        self.data_dir = data_dir
        
        testdata_path = Path(self.data_dir) / "testdata.json"
        metadata = json.load(testdata_path.open())
        
        self.data = metadata["utterances"]
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        utterance = self.data[index]
        feature_path = utterance["feature_path"]
        mel = torch.load(os.path.join(self.data_dir, feature_path))
        
        return feature_path, mel
    
def inference_collate_batch(batch):
    feature_paths, mels = zip(*batch)
    return feature_paths, torch.stack(mels)

In [110]:
import csv

def get_config():
    config = {
        "data_dir": "./Dataset",
        "model_path": "./model.ckpt",
        "output_path": "./output.csv"
    }
    
    return config

def main(
    data_dir,
    model_path,
    output_path,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[Info] Device: {device}")
    
    mapping_path = Path(data_dir) / "mapping.json"
    mapping = json.load(mapping_path.open())
    
    dataset = InferenceDataset(data_dir)
    dataloader = DataLoader(
        dataset=dataset,
        batch_size=1, # Predict speaker sample by sample in inference
        shuffle=False,
        drop_last=False,
        num_workers=8,
        collate_fn=inference_collate_batch
    )
    print("[Info] Finish loading data")
    
    speaker_num = len(mapping["id2speaker"])
    model = Classifier(n_spks=speaker_num).to(device)
    model.load_state_dict(torch.load(model_path))
    model.eval()
    print("[Info] Create a model")
    
    results = [["Id", "Category"]]
    for feature_paths, mels in tqdm(dataloader):
        with torch.no_grad():
            mels = mels.to(device)
            output = model(mels)
            preds = output.argmax(1).cpu().numpy()
            for feature_path, pred in zip(feature_paths, preds):
                results.append([feature_path, mapping["id2speaker"][str(pred)]])
                
    with open(output_path, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerows(results)
        
if __name__ == "__main__":
    main(**get_config())

[Info] Device: cuda
[Info] Finish loading data
[Info] Create a model


/tmp/ipykernel_2990857/1232978545.py:14: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = nn.TransformerEncoder(


  0%|          | 0/6000 [00:00<?, ?it/s]